# Filtered Resource Stats

Run `resource_filter.py` first if the filtered CSV files are missing or stale.

In [1]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path('data/benchmark') if Path('data/benchmark').exists() else Path('../benchmark')
raw = pd.read_csv(DATA_DIR / 'filtered_resources_raw.csv')
tagged = pd.read_csv(DATA_DIR / 'filtered_resources_tagged.csv')
raw.shape, tagged.shape

((4203, 19), (4203, 16))

In [2]:
pd.DataFrame({
    'dataset': ['raw', 'tagged'],
    'rows': [len(raw), len(tagged)],
    'columns': [len(raw.columns), len(tagged.columns)],
})

,dataset,rows,columns
0,raw,4203,19
1,tagged,4203,16


In [3]:
def split_pipe(series):
    return series.fillna('').astype(str).str.split('|').explode().loc[lambda s: s.ne('')]

def top_counts(series, n=30):
    return series.value_counts().rename_axis('value').reset_index(name='count').head(n)

pd.Series({
    'unique_resources': tagged['resource_id'].nunique(),
    'unique_categories': split_pipe(tagged['service_categories']).nunique(),
    'unique_counties': split_pipe(tagged['service_area']).nunique(),
    'unique_cities': tagged['city'].nunique(),
    'unique_zipcodes': tagged['zipcode'].nunique(),
}).to_frame('count')

,count
unique_resources,4203
unique_categories,34
unique_counties,92
unique_cities,382
unique_zipcodes,480


In [4]:
top_counts(split_pipe(tagged['service_categories']))

,value,count
0,Food,814
1,Information and Referral,529
2,Material Goods,491
3,Housing and Shelter,324
4,Government Offices and Public Services,307
5,Family and Caregiver Support,303
6,Police and Public Safety,302
7,Mental Health and Peer Support,239
8,Mental Health Treatment,187
9,Substance Use Help,174


In [5]:
top_counts(split_pipe(tagged['service_area']))

,value,count
0,MARION,535
1,ALLEN,243
2,LAKE,209
3,ST. JOSEPH,166
4,VANDERBURGH,133
5,ELKHART,131
6,MONROE,118
7,DELAWARE,110
8,MADISON,98
9,TIPPECANOE,95


In [6]:
pd.concat({
    'schedule_status': top_counts(tagged['schedule_status']),
    'intake_methods': top_counts(split_pipe(tagged['intake_methods'])),
    'document_requirements': top_counts(split_pipe(tagged['document_requirements'])),
}, names=['field']).reset_index(level=0)

,field,value,count
0,schedule_status,structured,4203
0,intake_methods,call,3149
1,intake_methods,walk_in,1758
2,intake_methods,online,994
3,intake_methods,appointment,912
4,intake_methods,mail,84
5,intake_methods,text,55
6,intake_methods,email,5
0,document_requirements,none,2608
1,document_requirements,photo_id,1443


In [7]:
windows = tagged['schedule_windows'].fillna('[]').apply(json.loads)
pd.Series({
    'resources_with_windows': windows.map(bool).sum(),
    'total_windows': windows.map(len).sum(),
    'avg_windows_per_resource': windows.map(len).mean(),
}).to_frame('value')

,value
resources_with_windows,4203.000000
total_windows,19379.000000
avg_windows_per_resource,4.610754
